# Financial Markets SQL Datamart

Inserire il testo per spiegare il tutto, maxi riassunto generale

In [1]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()


def get_connection():
    return psycopg2.connect(
        host=os.getenv("PGHOST"),
        database=os.getenv("PGDATABASE"),
        user=os.getenv("PGUSER"),
        password=os.getenv("PGPASSWORD"),
        sslmode=os.getenv("PGSSLMODE")
    )


def execute_query(query):
    conn = None
    cur = None

    try:
        conn = get_connection()
        cur = conn.cursor()

        cur.execute(query)
        conn.commit()

        print("✓ Sweet! Data updated")

    except Exception as e:
        if conn:
            conn.rollback()

        print("Errore:", e)

    finally:
        if cur:
            cur.close()

        if conn:
            conn.close()


In [2]:
# CREAZIONE DEGLI SCHEMAS DEL PROGGETTO

# si provede con la creazione degli scheme per:

True

#Creazione degli schemas del progetto

creazione degli schemas per:
* raw
* staging
* ref
* core
* mart
* audit

procedo con l'inserimento anche di mart, tuttavia al suo interno non saranno presenti delle tabelle, al fine di gestire al meglio gli spazi, questo schema sarà popolato in un secondo momento, ovvero quando le altre tabelle saranno funzionanti e popolate, in modo da valutare al meglio se procedere con un eventuale aumento della memoria disponibile del fornitore o non procedere con l'upgrade

In [5]:
query = """
CREATE SCHEMA IF NOT EXISTS raw;
CREATE SCHEMA IF NOT EXISTS staging;
CREATE SCHEMA IF NOT EXISTS ref;
CREATE SCHEMA IF NOT EXISTS core;
CREATE SCHEMA IF NOT EXISTS mart;
CREATE SCHEMA IF NOT EXISTS audit;
"""

execute_query(query)
print ("schemas create con sucesso")


schemas create con sucesso


Ora si procede con la creazione delle prime tabelle, sullo schema REF.


In [ ]:
query = """
CREATE TABLE IF NOT EXISTS ref.currencies (
    currency_code       CHAR(3)         PRIMARY KEY,
    currency_name       VARCHAR(50)     NOT NULL,
    currency_symbol     VARCHAR(8),
    decimal_places      SMALLINT        NOT NULL,
    is_active           BOOLEAN         NOT NULL,

    CONSTRAINT chk_currency_code_upper
        CHECK (currency_code = UPPER(currency_code)),

    CONSTRAINT chk_currency_decimal_places
        CHECK (decimal_places >= 0)
);
"""

execute_query(query)


In [23]:
# creazione della tabella Currencies
query = """
INSERT INTO ref.currencies (
    currency_code,
    currency_name,
    currency_symbol,
    decimal_places,
    is_active
)
VALUES
    ('EUR', 'Euro', '€', 2, TRUE),
    ('USD', 'US Dollar', '$', 2, TRUE),
    ('GBP', 'British Pound', '£', 2, TRUE),
    ('CHF', 'Swiss Franc', 'CHF', 2, TRUE)
ON CONFLICT (currency_code) DO NOTHING;
"""
execute_query(query)
print("Tabella popolata con successo")

Operazione completata con successo
Tabella popolata con successo


In [24]:
#Creazione della tabella Calendar
query = """
CREATE TABLE IF NOT EXISTS ref.calendar (
    date_key            INTEGER			PRIMARY KEY,
    full_date	        DATE			NOT NULL,
    year_number	        SMALLINT		NOT NULL,
    quarter_number      SMALLINT		NOT NULL,
    month_number	    SMALLINT		NOT NULL,
    month_name	        VARCHAR(15)		NOT NULL,
    week_number	        SMALLINT		NOT NULL,
    day_of_month	    SMALLINT		NOT NULL,
    day_of_week	        SMALLINT		NOT NULL,
    is_weekend	        BOOLEAN			NOT NULL,
    is_month_end	    BOOLEAN			NOT NULL,
    is_quarter_end	    BOOLEAN			NOT NULL,
    is_year_end	        BOOLEAN			NOT NULL
)
"""
execute_query(query)

Operazione completata con successo


In [9]:
#REF.countries
query = """
CREATE TABLE IF NOT EXISTS ref.countries (
    country_code        CHAR(2)	        PRIMARY KEY,
    country_name	    VARCHAR(100)	NOT NULL,
    region_name	        VARCHAR(50),
    is_active	        BOOLEAN			NOT NULL
)
"""
execute_query(query)

✓ Sweet! Data updated


In [10]:
#REF.exchanges
query = """
CREATE TABLE IF NOT EXISTS ref.exchanges (
    exchange_mic	        CHAR(4)	        PRIMARY KEY,
    exchange_name	        VARCHAR(120)	NOT NULL,
    country_code	        CHAR(2),
    timezone_name	        VARCHAR(64),
    default_currency_code	CHAR(3),
    is_active	            BOOLEAN			NOT NULL,

    CONSTRAINT fk_country FOREIGN KEY (country_code)
        REFERENCES ref.countries(country_code),
    CONSTRAINT fk_currency FOREIGN KEY (default_currency_code)
        REFERENCES ref.currencies(currency_code)
)
"""
execute_query(query)

✓ Sweet! Data updated


In [12]:
# CORE.data_sources
query = """
CREATE TABLE IF NOT EXISTS Core.data_sources (
    source_id	            BIGINT GENERATED ALWAYS AS IDENTITY	    PRIMARY KEY,
    source_code	            VARCHAR(30)			                    NOT NULL	UNIQUE,
    source_name	            VARCHAR(120)			                NOT NULL,
    source_category	        VARCHAR(30)			                    NOT NULL,
    base_url	            TEXT,
    access_method	        VARCHAR(20)			                    NOT NULL,
    authentication_type	    VARCHAR(20)			                    NOT NULL,
    license_notes	        TEXT,
    redistribution_allowed	BOOLEAN			                        NOT NULL,
    rate_limit_notes	    VARCHAR(200),
    is_official	            BOOLEAN			                        NOT NULL,
    is_active	            BOOLEAN			                        NOT NULL,
    last_verified_at	    TIMESTAMPTZ
)
"""
execute_query(query)

✓ Sweet! Data updated


In [17]:
#CORE.clients
query = """
CREATE TABLE IF NOT EXISTS Core.clients (
    client_id	        BIGINT GENERATED ALWAYS AS IDENTITY			PRIMARY KEY,
    client_code	        VARCHAR(30)			                        NOT NULL UNIQUE,
    client_name	        VARCHAR(150)			                    NOT NULL,
    client_type	        VARCHAR(30)			                        NOT NULL  CHECK (client_type IN ('RETAIL','PRIVATE','INSTITUTIONAL','CORPORATE', 'OTHER')),
    segment	            VARCHAR(30),
    country_code	    CHAR(2),
    risk_profile	    VARCHAR(30),
    onboarding_date	    DATE			                            NOT NULL,
    status	            VARCHAR(20)			                        NOT NULL    CHECK ( status IN ('ACTIVE','INACTIVE','CLOSED')),
    created_at	        TIMESTAMPTZ			                        NOT NULL    DEFAULT CURRENT_TIMESTAMP,
    updated_at	        TIMESTAMPTZ,

    CONSTRAINT fk_country FOREIGN KEY (country_code)
        REFERENCES ref.countries(country_code)
)
"""
execute_query(query)

✓ Sweet! Data updated


In [19]:
#CORE.issuer
# ISSUER
query = """
CREATE TABLE IF NOT EXISTS Core.issuers (
    issuer_id	    BIGINT GENERATED ALWAYS AS IDENTITY 	PRIMARY KEY,
    issuer_code	    VARCHAR(30)			NOT NULL	                UNIQUE,
    issuer_name	    VARCHAR(150)		NOT NULL,
    issuer_type	    VARCHAR(30)			NOT NULL,
    country_code	CHAR(2),
    sector	        VARCHAR(50),
    industry	    VARCHAR(80),
    lei_code	    CHAR(20)			UNIQUE,
    is_active	    BOOLEAN			    NOT NULL,

    CONSTRAINT fk_country FOREIGN KEY (country_code)
        REFERENCES ref.countries(country_code)

)
"""
execute_query(query)

✓ Sweet! Data updated


In [24]:
#RAW.INSTURMENT_PRICE
# INSTRUMENT_PRICE
query = """
CREATE TABLE raw.instrument_prices (
    -- Chiave tecnica
    raw_price_id           BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Identificativi strumento
    ticker                 VARCHAR(30),
    isin                   CHAR(12),

    -- Data del prezzo
    price_date             DATE NOT NULL,

    -- Prezzi
    open_price             NUMERIC(20,8),
    high_price             NUMERIC(20,8),
    low_price              NUMERIC(20,8),
    close_price            NUMERIC(20,8),
    adjusted_close         NUMERIC(20,8),

    -- Volume
    volume                 BIGINT,

    -- Valuta
    currency               CHAR(3),

    -- Lineage tecnico / metadati
    source_name            VARCHAR(100),
    source_file            VARCHAR(255),
    batch_id               BIGINT,
    loaded_at              TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);
CREATE INDEX IF NOT EXISTS idx_raw_instrument_prices_lookup
ON raw.instrument_prices (ticker, price_date);

CREATE INDEX IF NOT EXISTS idx_raw_instrument_prices_isin
ON raw.instrument_prices (isin);
"""
execute_query(query)

✓ Sweet! Data updated


In [6]:
#RAW.fx_rates
# FOREX RATES
query = """
CREATE TABLE raw.fx_rates (
    rate_date               DATE NOT NULL,
    base_currency           CHAR(3) NOT NULL,
    quote_currency          CHAR(3) NOT NULL,
    fx_rate                 NUMERIC(20, 10) NOT NULL,
    rate_type               VARCHAR(20) DEFAULT 'CLOSE',
    source_name             VARCHAR(50),
    ingestion_file_id       BIGINT,
    load_at                 TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT chk_raw_fx_rate CHECK (fx_rate > 0)
);

CREATE INDEX idx_raw_fx_rates_lookup
ON raw.fx_rates (rate_date, base_currency, quote_currency);
"""
execute_query(query)

✓ Sweet! Data updated


In [7]:
#RAW.TRANSACTIONS
# TRANSACTIONS
query = """
CREATE TABLE raw.transactions (
    -- Chiavi tecniche e identificativi
    raw_transaction_id     BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    source_transaction_id  VARCHAR(100),

    -- Dettagli Portafoglio e Cliente
    client_code            VARCHAR(50),
    account_code           VARCHAR(50),
    portfolio_code         VARCHAR(50),

    -- Date di Business
    transaction_date       DATE,
    settlement_date        DATE,
    trade_date             DATE,

    -- Anagrafica Strumento
    isin                   CHAR(12),
    ticker                 VARCHAR(30),
    source_instrument_id   VARCHAR(100),

    -- Dettagli Operazione
    transaction_type       VARCHAR(50),
    quantity               NUMERIC(24,8),
    price                  NUMERIC(20,8),

    -- Importi e Costi
    gross_amount           NUMERIC(24,8),
    net_amount             NUMERIC(24,8),
    commission             NUMERIC(20,8),
    fees                   NUMERIC(20,8),
    taxes                  NUMERIC(20,8),

    -- Valuta e FX
    currency               CHAR(3),
    fx_rate                NUMERIC(20,10),
    cash_amount            NUMERIC(24,8),

    -- Altri dettagli
    counterparty           VARCHAR(100),
    market                 VARCHAR(50),
    description            TEXT,

    -- Lineage tecnico / Meta-dati di caricamento
    source_name            VARCHAR(100),
    source_file            VARCHAR(255),
    batch_id               BIGINT,
    loaded_at              TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)


✓ Sweet! Data updated


In [8]:
#RAW.orporate_actions
# orporate_actions
query = """
CREATE TABLE raw.corporate_actions (
    -- Chiavi tecniche e identificativi
    raw_corporate_action_id  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    source_action_id         VARCHAR(100),

    -- Anagrafica Strumento
    isin                     CHAR(12),
    ticker                   VARCHAR(30),
    source_instrument_id     VARCHAR(100),

    -- Dettagli Evento
    action_type              VARCHAR(50),

    -- Date di Business
    announcement_date        DATE,
    ex_date                  DATE,
    record_date              DATE,
    payment_date             DATE,

    -- Importi e Rapporti
    currency                 CHAR(3),
    cash_amount              NUMERIC(20,8),
    dividend_per_share       NUMERIC(20,8),
    ratio_from               NUMERIC(18,8),
    ratio_to                 NUMERIC(18,8),

    -- Dati per cambi strumento (es. Merger / Ticker change)
    new_isin                 CHAR(12),
    new_ticker               VARCHAR(30),

    -- Altri dettagli
    description              TEXT,

    -- Lineage tecnico / Meta-dati di caricamento
    source_name              VARCHAR(100),
    source_file              VARCHAR(255),
    batch_id                 BIGINT,
    loaded_at                TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)

✓ Sweet! Data updated


In [9]:
#RAW.benchmarks
# benchmarks
query = """
CREATE TABLE raw.benchmarks (
    -- Chiavi tecniche e identificativi
    raw_benchmark_id     BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    source_benchmark_id  VARCHAR(100),

    -- Dettagli Benchmark
    benchmark_code       VARCHAR(30),
    benchmark_name       VARCHAR(150),

    -- Identificativi di mercato
    ticker               VARCHAR(30),
    isin                 CHAR(12),

    -- Tipologia e classificazione
    benchmark_type       VARCHAR(30),
    currency             CHAR(3),
    country_code         CHAR(2),
    provider             VARCHAR(50),
    asset_class          VARCHAR(40),

    -- Altri dettagli descrittivi
    description          TEXT,

    -- Lineage tecnico / Meta-dati di caricamento
    source_name          VARCHAR(100),
    source_file          VARCHAR(255),
    batch_id             BIGINT,
    loaded_at            TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)

✓ Sweet! Data updated


In [10]:
#RAW.benchmark_prices
# benchmark_prices
query = """
CREATE TABLE IF NOT EXISTS raw.benchmark_prices (
    -- Chiavi tecniche
    raw_benchmark_price_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Riferimenti Benchmark
    benchmark_code         VARCHAR(30),
    benchmark_ticker       VARCHAR(30),

    -- Data di osservazione
    price_date             DATE,

    -- Valori / Livelli Indice (OHLC)
    open_value             NUMERIC(20,8),
    high_value             NUMERIC(20,8),
    low_value              NUMERIC(20,8),
    close_value            NUMERIC(20,8),
    adjusted_close_value   NUMERIC(20,8),

    -- Valuta
    currency               CHAR(3),

    -- Lineage tecnico / Meta-dati di caricamento
    source_name            VARCHAR(100),
    source_file            VARCHAR(255),
    batch_id               BIGINT,
    loaded_at              TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""

execute_query(query)

✓ Sweet! Data updated


In [11]:
#RAW.positions
# positions
query = """
CREATE TABLE IF NOT EXISTS raw.positions (
    -- Chiavi tecniche
    raw_position_id        BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Data di validità del saldo
    position_date          DATE,

    -- Riferimenti Portafoglio e Cliente
    client_code            VARCHAR(50),
    portfolio_code         VARCHAR(50),
    account_code           VARCHAR(50),

    -- Identificativi Strumento
    isin                   CHAR(12),
    ticker                 VARCHAR(30),
    source_instrument_id   VARCHAR(100),

    -- Quantità
    quantity               NUMERIC(24,8),

    -- Valorizzazione di Mercato
    market_price           NUMERIC(20,8),
    market_value           NUMERIC(24,8),

    -- Valorizzazione di Costo
    cost_price             NUMERIC(20,8),
    cost_value             NUMERIC(24,8),

    -- Profitti e Perdite (P&L)
    unrealized_pnl         NUMERIC(24,8),

    -- Valuta
    currency               CHAR(3),

    -- Lineage tecnico / Meta-dati di caricamento
    source_name            VARCHAR(100),
    source_file            VARCHAR(255),
    batch_id               BIGINT,
    loaded_at              TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)

✓ Sweet! Data updated


Inserimento tabelle sullo schema STAGING

In [12]:
query = """
CREATE TABLE IF NOT EXISTS staging.instrument_prices (
    -- Chiave tecnica STAGING
    staging_price_id      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage verso RAW
    raw_price_id          BIGINT NOT NULL UNIQUE,

    -- Identificativi strumento normalizzati
    ticker                VARCHAR(30),
    isin                  VARCHAR(12),

    -- Data convertita
    price_date            DATE NOT NULL,

    -- Prezzi convertiti
    open_price            NUMERIC(20,8),
    high_price            NUMERIC(20,8),
    low_price             NUMERIC(20,8),
    close_price           NUMERIC(20,8),
    adjusted_close        NUMERIC(20,8),

    -- Volume
    volume                BIGINT,

    -- Valuta
    currency              CHAR(3),

    -- Fonte
    source_name           VARCHAR(100),

    -- Lineage temporale
    raw_loaded_at         TIMESTAMP,
    staged_at             TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)

✓ Sweet! Data updated


In [13]:
query = """
CREATE TABLE IF NOT EXISTS staging.instrument_prices (
    -- Chiave tecnica STAGING
    staging_price_id      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage verso RAW
    raw_price_id          BIGINT NOT NULL UNIQUE,

    -- Identificativi strumento normalizzati
    ticker                VARCHAR(30),
    isin                  VARCHAR(12),

    -- Data convertita
    price_date            DATE NOT NULL,

    -- Prezzi convertiti
    open_price            NUMERIC(20,8),
    high_price            NUMERIC(20,8),
    low_price             NUMERIC(20,8),
    close_price           NUMERIC(20,8),
    adjusted_close        NUMERIC(20,8),

    -- Volume
    volume                BIGINT,

    -- Valuta
    currency              CHAR(3),

    -- Fonte
    source_name           VARCHAR(100),

    -- Lineage temporale
    raw_loaded_at         TIMESTAMP,
    staged_at             TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)

✓ Sweet! Data updated


In [14]:
query = """
CREATE TABLE IF NOT EXISTS staging.transactions (
    staging_transaction_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_transaction_id       BIGINT NOT NULL UNIQUE,

    -- Identificativo sorgente
    source_transaction_id    VARCHAR(100),

    -- Cliente / Portafoglio
    client_code              VARCHAR(50),
    account_code             VARCHAR(50),
    portfolio_code           VARCHAR(50),

    -- Date
    transaction_date         DATE,
    settlement_date          DATE,
    trade_date               DATE,

    -- Strumento
    isin                     VARCHAR(12),
    ticker                   VARCHAR(30),
    source_instrument_id     VARCHAR(100),

    -- Operazione normalizzata
    transaction_type         VARCHAR(50),

    quantity                 NUMERIC(24,8),
    price                    NUMERIC(20,8),

    -- Importi
    gross_amount             NUMERIC(24,8),
    net_amount               NUMERIC(24,8),

    commission               NUMERIC(20,8),
    fees                     NUMERIC(20,8),
    taxes                    NUMERIC(20,8),

    -- Valuta
    currency                 CHAR(3),
    fx_rate                  NUMERIC(20,10),

    cash_amount              NUMERIC(24,8),

    -- Informazioni operative
    counterparty             VARCHAR(100),
    market                   VARCHAR(50),
    description              TEXT,

    -- Lineage
    source_name              VARCHAR(100),
    source_file              VARCHAR(255),
    batch_id                 BIGINT,
    raw_loaded_at            TIMESTAMPTZ,

    staged_at                TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_transactions_isin
ON staging.transactions (isin);

CREATE INDEX IF NOT EXISTS idx_stg_transactions_portfolio
ON staging.transactions (portfolio_code);

CREATE INDEX IF NOT EXISTS idx_stg_transactions_trade_date
ON staging.transactions (trade_date);
"""
execute_query(query)

✓ Sweet! Data updated


In [15]:
query = """
CREATE TABLE IF NOT EXISTS staging.corporate_actions (
    staging_corporate_action_id  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_corporate_action_id      BIGINT NOT NULL UNIQUE,

    source_action_id             VARCHAR(100),

    -- Strumento
    isin                         VARCHAR(12),
    ticker                       VARCHAR(30),
    source_instrument_id         VARCHAR(100),

    -- Evento normalizzato
    action_type                  VARCHAR(50),

    -- Date
    announcement_date            DATE,
    ex_date                      DATE,
    record_date                  DATE,
    payment_date                 DATE,

    -- Importi
    currency                     CHAR(3),
    cash_amount                  NUMERIC(20,8),
    dividend_per_share           NUMERIC(20,8),

    ratio_from                   NUMERIC(18,8),
    ratio_to                     NUMERIC(18,8),

    -- Eventuale nuovo strumento
    new_isin                     VARCHAR(12),
    new_ticker                   VARCHAR(30),

    description                  TEXT,

    -- Lineage
    source_name                  VARCHAR(100),
    source_file                  VARCHAR(255),
    batch_id                     BIGINT,
    raw_loaded_at                TIMESTAMPTZ,

    staged_at                    TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_corporate_actions_isin
ON staging.corporate_actions (isin);

CREATE INDEX IF NOT EXISTS idx_stg_corporate_actions_ex_date
ON staging.corporate_actions (ex_date);
"""
execute_query(query)

✓ Sweet! Data updated


In [16]:
query = """
CREATE TABLE IF NOT EXISTS staging.corporate_actions (
    staging_corporate_action_id  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_corporate_action_id      BIGINT NOT NULL UNIQUE,

    source_action_id             VARCHAR(100),

    -- Strumento
    isin                         VARCHAR(12),
    ticker                       VARCHAR(30),
    source_instrument_id         VARCHAR(100),

    -- Evento normalizzato
    action_type                  VARCHAR(50),

    -- Date
    announcement_date            DATE,
    ex_date                      DATE,
    record_date                  DATE,
    payment_date                 DATE,

    -- Importi
    currency                     CHAR(3),
    cash_amount                  NUMERIC(20,8),
    dividend_per_share           NUMERIC(20,8),

    ratio_from                   NUMERIC(18,8),
    ratio_to                     NUMERIC(18,8),

    -- Eventuale nuovo strumento
    new_isin                     VARCHAR(12),
    new_ticker                   VARCHAR(30),

    description                  TEXT,

    -- Lineage
    source_name                  VARCHAR(100),
    source_file                  VARCHAR(255),
    batch_id                     BIGINT,
    raw_loaded_at                TIMESTAMPTZ,

    staged_at                    TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_corporate_actions_isin
ON staging.corporate_actions (isin);

CREATE INDEX IF NOT EXISTS idx_stg_corporate_actions_ex_date
ON staging.corporate_actions (ex_date);
"""
execute_query(query)

✓ Sweet! Data updated


In [17]:
query = """
CREATE TABLE IF NOT EXISTS staging.benchmarks (
    staging_benchmark_id    BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_benchmark_id        BIGINT NOT NULL UNIQUE,

    source_benchmark_id     VARCHAR(100),

    -- Benchmark
    benchmark_code          VARCHAR(30) NOT NULL,
    benchmark_name          VARCHAR(150) NOT NULL,

    -- Identificativi
    ticker                  VARCHAR(30),
    isin                    VARCHAR(12),

    -- Classificazione
    benchmark_type          VARCHAR(30),
    currency                CHAR(3),
    country_code            CHAR(2),

    provider                VARCHAR(50),
    asset_class             VARCHAR(40),

    description             TEXT,

    -- Lineage
    source_name             VARCHAR(100),
    source_file             VARCHAR(255),
    batch_id                BIGINT,
    raw_loaded_at           TIMESTAMPTZ,

    staged_at               TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_benchmarks_code
ON staging.benchmarks (benchmark_code);
"""
execute_query(query)

✓ Sweet! Data updated


In [18]:
query = """
CREATE TABLE IF NOT EXISTS staging.benchmark_prices (
    staging_benchmark_price_id  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_benchmark_price_id      BIGINT NOT NULL UNIQUE,

    -- Benchmark
    benchmark_code              VARCHAR(30),
    benchmark_ticker            VARCHAR(30),

    -- Data
    price_date                  DATE NOT NULL,

    -- Valori
    open_value                  NUMERIC(20,8),
    high_value                  NUMERIC(20,8),
    low_value                   NUMERIC(20,8),
    close_value                 NUMERIC(20,8),
    adjusted_close_value        NUMERIC(20,8),

    currency                    CHAR(3),

    -- Lineage
    source_name                 VARCHAR(100),
    source_file                 VARCHAR(255),
    batch_id                    BIGINT,
    raw_loaded_at               TIMESTAMPTZ,

    staged_at                   TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_benchmark_prices_lookup
ON staging.benchmark_prices (
    benchmark_code,
    price_date
);
"""
execute_query(query)

✓ Sweet! Data updated


In [19]:
query = """
CREATE TABLE IF NOT EXISTS staging.positions (
    staging_position_id      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_position_id          BIGINT NOT NULL UNIQUE,

    -- Data posizione
    position_date            DATE NOT NULL,

    -- Cliente / portafoglio
    client_code              VARCHAR(50),
    portfolio_code           VARCHAR(50),
    account_code             VARCHAR(50),

    -- Strumento
    isin                     VARCHAR(12),
    ticker                   VARCHAR(30),
    source_instrument_id     VARCHAR(100),

    -- Quantità
    quantity                 NUMERIC(24,8),

    -- Valore di mercato
    market_price             NUMERIC(20,8),
    market_value             NUMERIC(24,8),

    -- Costo
    cost_price               NUMERIC(20,8),
    cost_value               NUMERIC(24,8),

    -- P&L
    unrealized_pnl           NUMERIC(24,8),

    currency                 CHAR(3),

    -- Lineage
    source_name              VARCHAR(100),
    source_file              VARCHAR(255),
    batch_id                 BIGINT,
    raw_loaded_at            TIMESTAMPTZ,

    staged_at                TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_positions_portfolio_date
ON staging.positions (
    portfolio_code,
    position_date
);

CREATE INDEX IF NOT EXISTS idx_stg_positions_isin
ON staging.positions (isin);
"""
execute_query(query)

✓ Sweet! Data updated


In [20]:
query = """
CREATE TABLE IF NOT EXISTS staging.fx_rates (
    staging_fx_rate_id      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_fx_rate_id          BIGINT NOT NULL UNIQUE,

    -- Business data
    rate_date               DATE NOT NULL,
    base_currency           CHAR(3) NOT NULL,
    quote_currency          CHAR(3) NOT NULL,

    fx_rate                 NUMERIC(20,10) NOT NULL,

    rate_type               VARCHAR(20) NOT NULL DEFAULT 'CLOSE',

    -- Fonte
    source_name             VARCHAR(50),
    ingestion_file_id       BIGINT,

    -- Lineage temporale
    raw_loaded_at           TIMESTAMPTZ,
    staged_at               TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT chk_stg_fx_rate_positive
        CHECK (fx_rate > 0),

    CONSTRAINT chk_stg_fx_different_currencies
        CHECK (base_currency <> quote_currency)
);

CREATE INDEX IF NOT EXISTS idx_stg_fx_rates_lookup
ON staging.fx_rates (
    rate_date,
    base_currency,
    quote_currency
);
"""
execute_query(query)

✓ Sweet! Data updated
